# HyperAdapt: two-stage と E2E の分類性能・公平性

同一 seed の two-stage HyperAdapt と single-stage E2E HyperAdapt を同じ CheXpert test split で比較する。各 run はユーザー停止により未完了であり、保存された best validation AUROC checkpoint を使う。two-stage は epoch 11、E2E は epoch 41 の checkpoint であり、1 seed の観察である。

## 予測 cache と属性表

予測は `analysis/common/predictions.py` が作る cache を読む。全体性能は AUROC / BACC / recall_0 / F1、サブグループは age / race / gender / age × race、公平性は AUROC gap / BACC gap / Eopp0 / Eopp1 とする。race は White / Asian / Black に限定する。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils
from IPython.display import display
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score, roc_auc_score

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import STYLE_SHEET, split_csv
from analysis.common.predictions import cache_path, load_cache

plt.style.use(STYLE_SHEET)
PACKAGE = ROOT / "analysis/hyperadapt_two_stage_vs_e2e"
CACHE, RESULTS, FIGURES = (PACKAGE / name for name in ("cache", "results", "figures"))
SPLIT = "test"
MODELS = {
    "2-stage HyperAdapt": "20260928T071550Z-hyperadapt-chexpert-from-resnet-s43-1166",
    "E2E HyperAdapt": "20260928T071552Z-hyperadapt-chexpert-from-resnet-s43-c33f",
}
COLORS = {"2-stage HyperAdapt": "#F58518", "E2E HyperAdapt": "#54A24B"}
METRICS = ["auroc", "bacc", "recall_0", "f1"]
METRIC_TITLES = {"auroc": "AUROC", "bacc": "BACC", "recall_0": "Recall (class 0)", "f1": "F1"}
FAIRNESS_METRICS = ["auroc_gap", "bacc_gap", "eopp0", "eopp1"]
FAIRNESS_TITLES = {"auroc_gap": "AUROC gap", "bacc_gap": "BACC gap", "eopp0": "Eopp0", "eopp1": "Eopp1"}
RACE_LABELS = {0: "White", 2: "Asian", 3: "Black"}
GROUP_ORDERS = {
    "age": ["<65", ">=65"],
    "race": ["White", "Asian", "Black"],
    "gender": ["Male", "Female"],
    "age_race": [f"{age} × {race}" for age in ("<65", ">=65") for race in RACE_LABELS.values()],
}

In [ ]:
frame = pd.read_csv(split_csv("chexpert", SPLIT))
images = frame["image"].to_numpy(dtype=str)
demographics = pd.DataFrame(
    {
        "age": np.where(frame["age"] >= 65, ">=65", "<65"),
        "race": frame["race"].map(RACE_LABELS),
        "gender": frame["sex"].map({0: "Male", 1: "Female"}),
    }
)
demographics["age_race"] = demographics["age"] + " × " + demographics["race"]
demographics["age"] = demographics["age"].mask(frame["age_missing"].astype(bool) | frame["age"].isna())
demographics["race"] = demographics["race"].mask(frame["race_missing"].astype(bool))
demographics["gender"] = demographics["gender"].mask(frame["sex_missing"].astype(bool))
demographics["age_race"] = demographics["age_race"].mask(demographics[["age", "race"]].isna().any(axis=1))
caches = {method: load_cache(cache_path(CACHE, run_id, SPLIT), images) for method, run_id in MODELS.items()}
demographics.notna().sum()

## 指標の集計

`recall_0` は陰性クラスの recall（specificity）。`Eopp0` と `Eopp1` はそれぞれ群間の specificity と TPR の最大差であり、小さいほど望ましい。

In [ ]:
def score(target, probability, prediction):
    both_classes = (target == 0).any() and (target == 1).any()
    positive, negative = target == 1, target == 0
    return {
        "auroc": roc_auc_score(target, probability) if both_classes else np.nan,
        "bacc": balanced_accuracy_score(target, prediction) if both_classes else np.nan,
        "recall_0": recall_score(target, prediction, pos_label=0, zero_division=0),
        "f1": f1_score(target, prediction, zero_division=0),
        "n": len(target),
        "tpr": prediction[positive].mean() if positive.any() else np.nan,
        "tnr": (prediction[negative] == 0).mean() if negative.any() else np.nan,
    }


overall_rows, subgroup_rows, fairness_rows = [], [], []
for method, cached in caches.items():
    values = score(cached["target"], cached["probabilities"][:, 1], cached["predictions"])
    overall_rows.extend(
        {"model": method, "metric": metric, "value": values[metric], "n": values["n"]} for metric in METRICS
    )
    for attribute, order in GROUP_ORDERS.items():
        group_scores = []
        for group in order:
            index = demographics.index[demographics[attribute] == group].to_numpy()
            if not len(index):
                continue
            group_value = score(
                cached["target"][index], cached["probabilities"][index, 1], cached["predictions"][index]
            )
            group_scores.append(group_value)
            subgroup_rows.extend(
                {
                    "model": method,
                    "attribute": attribute,
                    "group": group,
                    "metric": metric,
                    "value": group_value[metric],
                    "n": group_value["n"],
                }
                for metric in METRICS
            )
        groups = pd.DataFrame(group_scores)
        gaps = {
            "auroc_gap": groups.auroc.max() - groups.auroc.min(),
            "bacc_gap": groups.bacc.max() - groups.bacc.min(),
            "eopp0": groups.tnr.max() - groups.tnr.min(),
            "eopp1": groups.tpr.max() - groups.tpr.min(),
        }
        fairness_rows.extend(
            {"model": method, "attribute": attribute, "metric": metric, "value": value}
            for metric, value in gaps.items()
        )

overall, subgroup, fairness = map(pd.DataFrame, (overall_rows, subgroup_rows, fairness_rows))
RESULTS.mkdir(exist_ok=True)
overall.to_csv(RESULTS / f"classification_performance_{SPLIT}.csv", index=False)
subgroup.to_csv(RESULTS / f"subgroup_performance_{SPLIT}.csv", index=False)
fairness.to_csv(RESULTS / f"fairness_metrics_{SPLIT}.csv", index=False)
display(overall.pivot(index="metric", columns="model", values="value"))

## 比較図

各図の棒は checkpoint ごとの単一 test split の値であり、seed 間の不確実性は示さない。

In [ ]:
FIGURES.mkdir(exist_ok=True)


def grouped_bars(ax, data, metric, title):
    attribute = data.attribute.iloc[0]
    groups = [group for group in GROUP_ORDERS[attribute] if group in set(data.group)]
    width, x = 0.8 / len(MODELS), np.arange(len(groups))
    for offset, method in enumerate(MODELS):
        values = [
            data[(data.model == method) & (data.group == group) & (data.metric == metric)].value.iloc[0]
            for group in groups
        ]
        ax.bar(x + (offset - 0.5) * width, values, width, label=method, color=COLORS[method])
    ax.set(xticks=x, xticklabels=groups, title=title, ylim=(0, 1))
    ax.grid(axis="y", alpha=0.3)


fig, axes = plt.subplots(1, 4, figsize=(14, 4), constrained_layout=True, sharey=True)
for ax, metric in zip(axes, METRICS):
    values = [overall[(overall.model == method) & (overall.metric == metric)].value.iloc[0] for method in MODELS]
    ax.bar(MODELS, values, color=[COLORS[method] for method in MODELS])
    ax.set(title=METRIC_TITLES[metric], ylim=(0, 1))
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Score")
fig.savefig(FIGURES / f"overall_classification_performance_{SPLIT}.png", dpi=180)
display(fig)
plt.close(fig)

for attribute in GROUP_ORDERS:
    data = subgroup[subgroup.attribute == attribute]
    fig, axes = plt.subplots(
        2, 2, figsize=(20 if attribute == "age_race" else 12, 8), constrained_layout=True, sharey=True
    )
    for ax, metric in zip(axes.flat, METRICS):
        grouped_bars(ax, data, metric, METRIC_TITLES[metric])
        ax.tick_params(axis="x", rotation=55 if attribute == "age_race" else 25)
    axes[0, 0].legend(fontsize=8)
    fig.savefig(FIGURES / f"{attribute}_subgroup_performance_{SPLIT}.png", dpi=180)
    display(fig)
    plt.close(fig)
    fair = fairness[fairness.attribute == attribute]
    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    width, x = 0.8 / len(MODELS), np.arange(len(FAIRNESS_METRICS))
    for offset, method in enumerate(MODELS):
        values = [fair[(fair.model == method) & (fair.metric == metric)].value.iloc[0] for metric in FAIRNESS_METRICS]
        ax.bar(x + (offset - 0.5) * width, values, width, label=method, color=COLORS[method])
    ax.set(
        xticks=x,
        xticklabels=[FAIRNESS_TITLES[metric] for metric in FAIRNESS_METRICS],
        ylabel="Gap",
        ylim=(0, 1),
        title=f"{attribute} fairness",
    )
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)
    fig.savefig(FIGURES / f"{attribute}_fairness_{SPLIT}.png", dpi=180)
    display(fig)
    plt.close(fig)